# Project: BBLF AI Selector v2 
# Section: Model Feature Creation
## Sub Section: Pre Tournament

Goal: To clean the raw player data set and create required features for model build

Things to add:
1. Individual Player Actual Cricket Statistics (current season and previous seasons)
2. Power Surge (somehow/ backpopulate)
3. Adjust average fields (if not enough data should be null e.g. 3 game average should be null until atleast 3 games)
4. T stat/ Sharpe Ratio of current and lag points
5. Need metric/ feature to assess players joining late in the tournament (usual bad players)


# 0. Prerequistes

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import OneHotEncoder

os.getcwd()
directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/'

# 1. Data Extraction

In [2]:
# 1a. Pull in all_matches csv file 

data_prep_df = pd.read_csv(os.path.join(directory,'python_data/fant_point_play_tbl.csv'), low_memory=False)

# 1b. Group & fix mis match venue names
unique_venues = data_prep_df['venue'].unique()
print(unique_venues)

data_prep_df['venue'] = data_prep_df['venue'].replace('Western Australia Cricket Association Ground', 'WACA')
data_prep_df['venue'] = data_prep_df['venue'].replace('W.A.C.A. Ground', 'WACA')
data_prep_df['venue'] = data_prep_df['venue'].replace('Brisbane Cricket Ground, Woolloongabba', 'GABBA')
data_prep_df['venue'] = data_prep_df['venue'].replace('Brisbane Cricket Ground', 'GABBA')
data_prep_df['venue'] = data_prep_df['venue'].replace('Brisbane Cricket Ground, Woolloongabba, Brisbane', 'GABBA')
data_prep_df['venue'] = data_prep_df['venue'].replace('Aurora Stadium', 'Launceston')
data_prep_df['venue'] = data_prep_df['venue'].replace('University of Tasmania Stadium, Launceston', 'Launceston')
data_prep_df['venue'] = data_prep_df['venue'].replace('Aurora Stadium, Launceston', 'Launceston')
data_prep_df['venue'] = data_prep_df['venue'].replace('Manuka Oval', 'Manuka')
data_prep_df['venue'] = data_prep_df['venue'].replace('Manuka Oval, Canberra', 'Manuka')
data_prep_df['venue'] = data_prep_df['venue'].replace('Docklands Stadium, Melbourne', 'Marvel')
data_prep_df['venue'] = data_prep_df['venue'].replace('Docklands Stadium', 'Marvel')
data_prep_df['venue'] = data_prep_df['venue'].replace('International Sports Stadium, Coffs Harbour', 'Coffs Harbour')
data_prep_df['venue'] = data_prep_df['venue'].replace('International Sports Stadium', 'Coffs Harbour')
data_prep_df['venue'] = data_prep_df['venue'].replace('GMHBA Stadium, South Geelong, Victoria', 'Geelong')
data_prep_df['venue'] = data_prep_df['venue'].replace('Geelong Cricket Ground', 'Geelong')
data_prep_df['venue'] = data_prep_df['venue'].replace('Simonds Stadium, South Geelong, Victoria', 'Geelong')
data_prep_df['venue'] = data_prep_df['venue'].replace('Sydney Cricket Ground', 'SCG')
data_prep_df['venue'] = data_prep_df['venue'].replace('Lavington Sports Oval, Albury', 'Albury')
data_prep_df['venue'] = data_prep_df['venue'].replace('North Sydney Oval, Sydney', 'North Sydney Oval')
data_prep_df['venue'] = data_prep_df['venue'].replace("Cazaly's Stadium, Cairns", 'Cairns')
data_prep_df['venue'] = data_prep_df['venue'].replace("Junction Oval, Melbourne", 'Junction Oval')
data_prep_df['venue'] = data_prep_df['venue'].replace("Sydney Showground Stadium", 'Sydney Showground')
data_prep_df['venue'] = data_prep_df['venue'].replace("Stadium Australia", 'Sydney Showground')
data_prep_df['venue'] = data_prep_df['venue'].replace("Traeger Park", 'Alice Springs')
data_prep_df['venue'] = data_prep_df['venue'].replace("Bellerive Oval, Hobart", 'Hobart')
data_prep_df['venue'] = data_prep_df['venue'].replace("Bellerive Oval", 'Hobart')
data_prep_df['venue'] = data_prep_df['venue'].replace("Ted Summerton Reserve", 'Moe')
data_prep_df['venue'] = data_prep_df['venue'].replace("Carrara Oval", 'Gold Coast')

data_prep_df['venue'] = data_prep_df['venue'].replace('WACA', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Launceston', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Coffs Harbour', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Geelong', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Albury', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('North Sydney Oval', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Cairns', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Junction Oval', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Alice Springs', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Moe', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Gold Coast', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Manuka', "Other")

unique_venues = data_prep_df['venue'].unique()
print(unique_venues)

unique_team = data_prep_df['team'].unique()
print(unique_team)

# 1c. Team Home and Away Flag
home_conditions = [
        (data_prep_df['venue'] == "Other"),
        (data_prep_df['venue'] == "GABBA") & (data_prep_df['team'] == "Brisbane Heat"),
        (data_prep_df['venue'] == "Melbourne Cricket Ground") & (data_prep_df['team'] == "Melbourne Stars"),
        (data_prep_df['venue'] == "Hobart") & (data_prep_df['team'] == "Hobart Hurricanes"),
        (data_prep_df['venue'] == "Sydney Showground") & (data_prep_df['team'] == "Sydney Thunder"),
        (data_prep_df['venue'] == "Adelaide Oval") & (data_prep_df['team'] == "Adelaide Strikers"),
        (data_prep_df['venue'] == "SCG") & (data_prep_df['team'] == "Sydney Sixers"),
        (data_prep_df['venue'] == "Marvel") & (data_prep_df['team'] == "Melbourne Renegades"),
        (data_prep_df['venue'] == "Perth Stadium") & (data_prep_df['team'] == "Perth Scorchers")
    ]

home_group = [0,1,1,1,1,1,1,1,1]

data_prep_df["Home_f"] = np.select(home_conditions, home_group)

['Western Australia Cricket Association Ground'
 'Brisbane Cricket Ground, Woolloongabba' 'Melbourne Cricket Ground'
 'Bellerive Oval' 'Sydney Showground Stadium' 'Adelaide Oval'
 'Sydney Cricket Ground' 'Docklands Stadium' 'Manuka Oval'
 'Stadium Australia' 'W.A.C.A. Ground'
 'Simonds Stadium, South Geelong, Victoria' 'Traeger Park'
 'Aurora Stadium' 'Perth Stadium' 'Carrara Oval' 'Geelong Cricket Ground'
 'Ted Summerton Reserve' 'International Sports Stadium'
 'Brisbane Cricket Ground' 'Manuka Oval, Canberra'
 'Docklands Stadium, Melbourne' 'Aurora Stadium, Launceston'
 'Bellerive Oval, Hobart'
 'Brisbane Cricket Ground, Woolloongabba, Brisbane'
 'Junction Oval, Melbourne' 'GMHBA Stadium, South Geelong, Victoria'
 'International Sports Stadium, Coffs Harbour' "Cazaly's Stadium, Cairns"
 'University of Tasmania Stadium, Launceston'
 'Lavington Sports Oval, Albury' 'North Sydney Oval, Sydney']
['Other' 'GABBA' 'Melbourne Cricket Ground' 'Hobart' 'Sydney Showground'
 'Adelaide Oval' 'SC

# 2. Current Season Fantasy Point

In [3]:
season_df = data_prep_df[["player", "fantasy_point", "season"]]
season_df_agg = season_df.groupby(['player', 'season'], as_index=False).agg(
season_fp =('fantasy_point',"sum"),
avg_season_fp = ('fantasy_point', "mean"),
max_season_fp = ('fantasy_point', 'max'),
min_season_fp = ('fantasy_point', 'min'),
med_season_fp = ('fantasy_point', 'median'),
sd_season_fp = ('fantasy_point', 'std'),
match_cnt = ('fantasy_point', 'count'))

# 3. Response Variable
1. Individual Game Points
2. Player Season Standard Deviation (less complex for simulation tool)

In [4]:
resp_df = data_prep_df[["match_id", "player", "season", "fantasy_point", "team", "opp"]].rename(columns = {"fantasy_point": "resp_var"})

# 4. Feature Creation

In [5]:
# 4a. Previous Season/s Fantasy Point Summary Stats 
# Lag 1
season_df_lag1_stat = season_df_agg.copy()
season_df_lag1_stat['season'] = season_df_lag1_stat['season'] + 1
season_df_lag1_stat = season_df_lag1_stat.rename(columns={c:c+'_lag1' for c in season_df_lag1_stat.columns if c not in ['player', 'season']})

fant_model_df = pd.merge(resp_df, season_df_lag1_stat, left_on = ["player", "season"], right_on = ["player", "season"], how = "left")

# Lag 2
season_df_lag2_stat = season_df_agg.copy()
season_df_lag2_stat['season'] = season_df_lag2_stat['season'] + 2
season_df_lag2_stat = season_df_lag2_stat.rename(columns={c:c+'_lag2' for c in season_df_lag2_stat.columns if c not in ['player', 'season']})

fant_model_df = pd.merge(fant_model_df, season_df_lag2_stat, left_on = ["player", "season"], right_on = ["player", "season"], how = "left")

# Lag 3
season_df_lag3_stat = season_df_agg.copy()
season_df_lag3_stat['season'] = season_df_lag3_stat['season'] + 3
season_df_lag3_stat = season_df_lag3_stat.rename(columns={c:c+'_lag3' for c in season_df_lag3_stat.columns if c not in ['player', 'season']})

fant_model_df = pd.merge(fant_model_df , season_df_lag3_stat, left_on = ["player", "season"], right_on = ["player", "season"], how = "left")

# 4b. Individual Match Features
# Match Venue
match_fact_df = data_prep_df[["match_id","player", "venue", "Home_f"]]
fant_model_df = pd.merge(fant_model_df , match_fact_df, left_on = ["match_id", "player"], right_on = ["match_id", "player"], how = "left")

# 4c. Opponent Season Power Ranking
team_rank_df = pd.read_csv(os.path.join(directory,'add_data_created/pre_tourny/team_season_rank.csv'), low_memory=False).rename(columns = {"team": "opp"})
fant_model_df = pd.merge(fant_model_df , team_rank_df, left_on = ["season", "opp"], right_on = ["season", "opp"], how = "left")

# 5. One-Hot Encoding Factor Features

In [6]:
# Create binary dummy variables
fant_model_df = pd.get_dummies(fant_model_df, columns= ['opp','venue','rank_group'])

# Convert dummies to object type
fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"] = fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"].astype(int)
fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"] = fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"].astype(object)

fant_model_df.loc[:,"venue_Adelaide Oval":"venue_Sydney Showground"] = fant_model_df.loc[:,"venue_Adelaide Oval":"venue_Sydney Showground"].astype(int)
fant_model_df.loc[:,"venue_Adelaide Oval":"venue_Sydney Showground"] = fant_model_df.loc[:,"venue_Adelaide Oval":"venue_Sydney Showground"].astype(object)

fant_model_df.loc[:,"rank_group_High":"rank_group_Middle"] = fant_model_df.loc[:,"rank_group_High":"rank_group_Middle"].astype(int)
fant_model_df.loc[:,"rank_group_High":"rank_group_Middle"] = fant_model_df.loc[:,"rank_group_High":"rank_group_Middle"].astype(object)

C:\Users\dilan\AppData\Local\Temp\ipykernel_29208\1765605496.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"] = fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"].astype(int)
C:\Users\dilan\AppData\Local\Temp\ipykernel_29208\1765605496.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"] = fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"].astype(int)
C:\Users\dilan\AppData\Local\Temp\ipykernel_29208\1765605496.py:5: FutureWarning: Setting an item of incompatible dtype is d

# 6. Interaction Variables

In [7]:
# 6a. Venue & Home Flag (See if the certain teams have bigger home ground advantage)
fant_model_df["Home_Adelaide Strikers"] = np.where((fant_model_df["venue_Adelaide Oval"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
fant_model_df["Home_Melbourne Stars"] = np.where((fant_model_df["venue_Melbourne Cricket Ground"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
fant_model_df["Home_Melbourne Renegades"] = np.where((fant_model_df["venue_Marvel"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
fant_model_df["Home_Brisbane Heat"] = np.where((fant_model_df["venue_GABBA"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
fant_model_df["Home_Perth Scorchers"] = np.where((fant_model_df["venue_Perth Stadium"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
fant_model_df["Home_Sydney Sixers"] = np.where((fant_model_df["venue_SCG"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
fant_model_df["Home_Sydney Thunder"] = np.where((fant_model_df["venue_Sydney Showground"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
fant_model_df["Home_Hobart Hurricanes"] = np.where((fant_model_df["venue_Hobart"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)

fant_model_df.loc[:,"Home_Adelaide Strikers":"Home_Hobart Hurricanes"] = fant_model_df.loc[:,"Home_Adelaide Strikers":"Home_Hobart Hurricanes"].astype(object)

C:\Users\dilan\AppData\Local\Temp\ipykernel_29208\3933861948.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fant_model_df.loc[:,"Home_Adelaide Strikers":"Home_Hobart Hurricanes"] = fant_model_df.loc[:,"Home_Adelaide Strikers":"Home_Hobart Hurricanes"].astype(object)
C:\Users\dilan\AppData\Local\Temp\ipykernel_29208\3933861948.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fant_model_df.loc[:,"Home_Adelaide Strikers":"Home_Hobart Hurricanes"] = fant_model_df.loc[:,"Home_Adelaide Strikers":"Home_Hobart Hurricanes"].astype(object)
C:\Users\dilan\AppData\Local\Temp\ipykernel_29208\3933861948.py:11: FutureWarning: Setting an 

# 7. Final Modelling Dataset

In [8]:
# Adjust 0 points to very small non 0 number 
fant_model_df["resp_var"] = fant_model_df["resp_var"].replace(0, 0.01)

# Remove first four season (due to lag variables up to 3 seasons prior. Also remove as over 10 years old)
fant_model_df = fant_model_df[fant_model_df["season"] > 4]    

# Save final model dataframe
fant_model_df.to_csv(os.path.join(directory,'python_data/bblf_v2_model_data.csv'))

# 8. Create BBL15 Previous Season Data Features for Model Scoring

In [ ]:
# 8a. Lag Variables (incl latest BBL data for players who have played BBL but the previous season)
# #season_df_lag1_stat_14 = season_df_lag1_stat[season_df_lag1_stat["season"] == 14]
# #season_df_lag2_stat_14 = season_df_lag2_stat[season_df_lag2_stat["season"] == 14]
# #season_df_lag3_stat_14 = season_df_lag3_stat[season_df_lag3_stat["season"] == 14]
# #
# #season_df_lag_14 = pd.merge(season_df_lag1_stat_14 , season_df_lag2_stat_14, left_on = ["player","season"], right_on = ["player","season"], how = "left")
# #season_df_lag_14 = pd.merge(season_df_lag_14 , season_df_lag3_stat_14, left_on = ["player","season"], right_on = ["player","season"], how = "left")

# # player_df = pd.read_csv(os.path.join(directory,'data/python_datasets/player_price.csv'), low_memory=False)
# # player_df = player_df[["player", "Full_Name"]]

#     # Latest BBL player records
#  # 1. For loop for each distinct player to find max season
 
# player_list = season_df_lag1_stat.player.unique()
# #ind_play_lag_df = season_df_lag1_stat[season_df_lag1_stat["player"] == "JW Wells"]
# #ind_play_last_season = ind_play_lag_df[["season"]].max()
# play_last_season = []

# for x in player_list:

#     # select individual player records 
#     ind_play_lag_df = season_df_lag1_stat[season_df_lag1_stat["player"] == x]
#     ind_play_last_season = ind_play_lag_df[["season"]].sort_values(by='season', ascending=False).iloc[0]
#     ind_play_last_season = max(ind_play_last_season)
#     play_last_season = play_last_season + [ind_play_last_season]

# join_list = {'player': player_list, 'season': play_last_season}
# print(join_list)

# bbl14_play_lags = pd.DataFrame(join_list)

# all_play_latest_season_lags = pd.merge(bbl14_play_lags , season_df_lag1_stat, left_on = ["player","season"], right_on = ["player","season"], how = "left")
# all_play_latest_season_lags = pd.merge(all_play_latest_season_lags , season_df_lag2_stat, left_on = ["player","season"], right_on = ["player","season"], how = "left")
# all_play_latest_season_lags = pd.merge(all_play_latest_season_lags , season_df_lag3_stat, left_on = ["player","season"], right_on = ["player","season"], how = "left")
# all_play_latest_season_lags = all_play_latest_season_lags.drop(["season"], axis = 1)
# print(all_play_latest_season_lags)

# # Join all_player_latest_season_lags to the bbl14 player list
# season_df_lag_14 = pd.merge(player_df, all_play_latest_season_lags, left_on = ["player"], right_on = ["player"], how = "left")


# season_df_lag_14.to_csv('data/python_datasets/test.csv')

# season_df_lag_14.to_csv('data/python_datasets/bbl14_lags.csv')